# Projekt 01 (basic) — Das Internet als Graph

> **Modul 16 — ML for Networks 2** · Format: **Jupyter Notebook**
>
> **Warum dieses Format?** Graphen versteht man durch **Hinschauen**: Gradverteilungen plotten,
> Hubs finden, das Netz zerbrechen lassen. Explorative Analyse mit vielen Plots — genau das
> Notebook-Format (wie Modul 02/15).

## Ziel
Du arbeitest mit der **echten Internet-Topologie** (AS-Level, Oregon Route-Views, März 2001 —
10 670 autonome Systeme, 22 002 Peerings) und lernst:

1. die **Gradverteilung** und warum das Internet **scale-free** ist,
2. **wie man den Power-Law-Exponenten richtig schätzt** — und warum der naheliegende Weg
   (log-log-Fit) **um Faktor 2 danebenliegt**,
3. **Zentralitäten** — wer ist wichtig, und nach welcher Definition,
4. die **Robustheit** des Internets: harmlos gegen Zufallsausfälle, **fatal** gegen Hub-Angriffe,
5. **Link Prediction** mit klassischen Heuristiken (Common Neighbors, Jaccard, **Adamic-Adar**,
   Preferential Attachment) — inkl. der zwei Fallen, die solche Ergebnisse ruinieren.

## Vorwissen
Skript Modul 16, Abschnitte **1** (Graph, Scale-Free, Robustheit) und **2.1** (Heuristiken).
`networkx`-Grundlagen sind nicht nötig — wir führen sie ein.

## Was am Ende funktionieren soll
Die Reproduktion zweier berühmter Ergebnisse **auf echten Daten**: der Power-Law-Exponent
$\alpha\approx2{,}1$ (Faloutsos³ 1999) und die Robustheits-Asymmetrie (Albert/Jeong/Barabási
2000). Plus ein Link-Predictor, der deutlich besser als Zufall ist.

> **Arbeitsweise:** `# TODO`-Stellen selbst füllen. Lösung: `loesung/internet_graph_loesung.ipynb`.

In [ ]:
import urllib.request, gzip, os, random
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

SEED = 0
rng = np.random.default_rng(SEED)
print("networkx", nx.__version__)

## 1 · Echte Daten holen

**Datensatz:** `oregon1_010331` aus dem **SNAP**-Repository (Stanford) — die AS-Peering-Topologie,
abgeleitet aus **BGP**-Daten der Oregon Route-Views vom 31. März 2001. Knoten = **autonomes
System** (ein Provider/eine Organisation mit eigener Routing-Politik), Kante = die beiden AS
**peeren** miteinander.

Das sind **echte Messdaten** — 69 KB, Download in ~1 s, danach lokal gecached.

> **Wichtiger Vorbehalt** (Skript 4.4): Die „echte" AS-Topologie kennt **niemand**.
> Route-Collectors sehen nur, was BGP ihnen zeigt; **Peerings zwischen kleinen AS fehlen
> systematisch**. Wir arbeiten also auf einem *unvollständigen, verzerrt gemessenen* Graphen —
> das gilt für **alle** Ergebnisse unten.

In [ ]:
URL = "https://snap.stanford.edu/data/oregon1_010331.txt.gz"
PFAD = "daten/oregon1_010331.txt.gz"
os.makedirs("daten", exist_ok=True)

if not os.path.exists(PFAD):
    req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})  # ohne UA: 403
    with urllib.request.urlopen(req, timeout=60) as r, open(PFAD, "wb") as f:
        f.write(r.read())
    print("heruntergeladen ->", PFAD)
else:
    print("bereits vorhanden ->", PFAD)

with gzip.open(PFAD, "rt") as f:
    zeilen = [z for z in f if not z.startswith("#")]

G = nx.parse_edgelist(zeilen, nodetype=int)
print(f"\nGraph: {G.number_of_nodes():,} Knoten (AS), {G.number_of_edges():,} Kanten (Peerings)")
print("zusammenhaengend:", nx.is_connected(G))

## 2 · Erste Kennzahlen

**Deine Aufgabe:** Berechne die Grad-Statistiken. Achte auf das Verhältnis **Median vs.
Mittelwert vs. Maximum** — dasselbe Muster wie bei den Flow-Größen in Modul 15 (schwerer Rand).

In [ ]:
# grade = numpy-Array aller Knotengrade.  Tipp: G.degree() liefert (knoten, grad)-Paare.
# TODO: grade = ...
grade = None
raise NotImplementedError

print(f"Grad  min {grade.min()} | median {np.median(grade):.0f} | "
      f"mittel {grade.mean():.2f} | max {grade.max()}")
print(f"Dichte: {nx.density(G):.6f}  ({100*nx.density(G):.4f} % der moeglichen Kanten)")
print(f"mittlerer Clustering-Koeffizient: {nx.average_clustering(G):.4f}")

## 3 · Die Gradverteilung — und die Falle

Trage die Gradverteilung **doppelt-logarithmisch** auf. Ein Potenzgesetz $P(d)\propto d^{-\alpha}$
wird im log-log-Plot zur **Geraden** — das ist der berühmte Faloutsos³-Befund (1999).

Die **CCDF** (komplementäre Verteilungsfunktion, $P(D\ge d)$) ist dabei die ehrlichere
Darstellung als das Histogramm: sie braucht kein Binning und der Schwanz rauscht weniger.

In [ ]:
werte, anzahl = np.unique(grade, return_counts=True)
pmf = anzahl / anzahl.sum()
ccdf = np.array([(grade >= w).mean() for w in werte])     # P(D >= d)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4))
ax1.loglog(werte, pmf, "o", ms=3, alpha=0.6)
ax1.set(xlabel="Grad d", ylabel="P(D = d)", title="Gradverteilung (PMF, log-log)")
ax2.loglog(werte, ccdf, "-", lw=2)
ax2.set(xlabel="Grad d", ylabel="P(D >= d)", title="CCDF (log-log) - der ehrlichere Plot")
for ax in (ax1, ax2):
    ax.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()
print("Beide ~ Gerade im log-log-Plot => Potenzgesetz => scale-free.")

### 3.1 · Den Exponenten schätzen — richtig und falsch ⚠️

Jetzt der wichtigste methodische Punkt des Projekts.

**Der naheliegende Weg:** Gerade durch die log-log-Punkte fitten (`np.polyfit`), Steigung
ablesen. **Das ist systematisch falsch.** Im Schwanz liegen pro Grad-Wert nur 0–3 Knoten; deren
logarithmiertes Rauschen ist massiv verzerrt, und leere Bins verschwinden komplett. Die
Kleinste-Quadrate-Gerade wird dadurch verbogen.

**Der richtige Weg** (Clauset, Shalizi & Newman 2009): der **Maximum-Likelihood-Schätzer**, nur
auf dem Schwanz $d\ge d_{\min}$:
$$\hat\alpha = 1 + n\Big[\sum_{i=1}^{n}\ln\frac{d_i}{d_{\min}-\tfrac12}\Big]^{-1}$$

**Deine Aufgabe:** Implementiere beide und vergleiche. Der Literaturwert für die
Internet-AS-Topologie ist $\alpha\approx 2{,}1$ — welche Methode trifft ihn?

In [ ]:
def alpha_naiv(werte, anzahl):
    # Naiver log-log-Fit auf der PMF. Rueckgabe: geschaetztes alpha (positiv).
    # TODO: np.polyfit(log(werte), log(pmf), 1) -> Steigung; alpha = -Steigung
    raise NotImplementedError

def alpha_mle(grade, d_min):
    # Clauset-MLE auf dem Schwanz d >= d_min.
    # TODO: x = grade[grade >= d_min]
    #       return 1 + len(x) / np.sum(np.log(x / (d_min - 0.5)))
    raise NotImplementedError

print(f"naiver log-log-Fit : alpha = {alpha_naiv(werte, anzahl):.2f}")
for d_min in (1, 2, 5, 10):
    print(f"MLE (d_min={d_min:2d})     : alpha = {alpha_mle(grade, d_min):.2f}  "
          f"(n={int((grade >= d_min).sum())})")
print("\nLiteraturwert Internet-AS: alpha ~ 2.1")

## 4 · Wer ist wichtig? Zentralitäten

Verschiedene Definitionen von „wichtig" — mit sehr unterschiedlichen Kosten:
- **Grad**: viele Nachbarn (billig, lokal).
- **Betweenness**: auf wie vielen kürzesten Pfaden liege ich? (**der Flaschenhals**) — teuer,
  $O(|V||E|)$, deshalb hier per Stichprobe (`k=200`).
- **PageRank**: wichtig ist, wer wichtige Nachbarn hat (rekursiv).

Vorgegeben — läuft ~10 s.

In [ ]:
grad_z = dict(G.degree())
btw = nx.betweenness_centrality(G, k=200, seed=SEED)   # Stichprobe: exakt waere zu teuer
pr = nx.pagerank(G)

def top(d, k=5):
    return sorted(d.items(), key=lambda t: -t[1])[:k]

print("Top-5 nach Grad       :", [(as_, g) for as_, g in top(grad_z)])
print("Top-5 nach Betweenness:", [(as_, round(v, 4)) for as_, v in top(btw)])
print("Top-5 nach PageRank   :", [(as_, round(v, 5)) for as_, v in top(pr)])

gemeinsam = len(set(dict(top(grad_z, 20)).keys()) & set(dict(top(btw, 20)).keys()))
print(f"\nUeberschneidung der Top-20 (Grad vs. Betweenness): {gemeinsam}/20")
print("-> Im Internet korrelieren die Masse stark: die grossen Hubs SIND die Flaschenhaelse.")

## 5 · Robustheit — der berühmteste Netzwerk-Befund 💥

**Albert, Jeong & Barabási (2000, Nature):** Scale-free-Netze sind **robust gegen zufällige
Ausfälle**, aber **fragil gegen gezielte Angriffe auf Hubs**.

**Deine Aufgabe:** Entferne schrittweise Knoten — einmal **zufällig** (Router fallen aus,
Kabelbagger), einmal **gezielt nach Grad** (ein Angreifer nimmt sich die größten AS vor) — und
miss den Anteil der Knoten in der **größten zusammenhängenden Komponente**.

In [ ]:
def groesste_komponente_anteil(H, n_original):
    # Anteil der Knoten in der groessten zusammenhaengenden Komponente.
    if H.number_of_nodes() == 0:
        return 0.0
    return len(max(nx.connected_components(H), key=len)) / n_original

n0 = G.number_of_nodes()
nach_grad = [k for k, _ in sorted(G.degree(), key=lambda t: -t[1])]   # Hubs zuerst
zufaellig = list(G.nodes()); random.Random(SEED).shuffle(zufaellig)

anteile = np.linspace(0, 0.10, 11)
kurve_zufall, kurve_angriff = [], []
for f in anteile:
    k = int(f * n0)
    # TODO: Fuer beide Reihenfolgen eine Kopie von G machen (G.copy()), die ersten k Knoten
    #       daraus entfernen (H.remove_nodes_from(...)) und groesste_komponente_anteil(H, n0)
    #       an kurve_zufall bzw. kurve_angriff anhaengen.
    raise NotImplementedError

for f, z, a in zip(anteile, kurve_zufall, kurve_angriff):
    print(f"{100*f:5.1f}% entfernt | Zufall {z:.3f} | gezielt {a:.3f}")

In [ ]:
plt.figure(figsize=(7.5, 4.6))
plt.plot(100*anteile, kurve_zufall, "o-", lw=2, label="zufaelliger Ausfall")
plt.plot(100*anteile, kurve_angriff, "s-", lw=2, color="crimson",
         label="gezielter Angriff (Hubs zuerst)")
plt.xlabel("entfernte Knoten [%]"); plt.ylabel("Anteil in groesster Komponente")
plt.title("Robustheit der echten Internet-Topologie"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Bei 5 % entfernt:  Zufall {kurve_zufall[5]:.3f}  vs.  gezielt {kurve_angriff[5]:.3f}")
print("-> 5 % der HUBS entfernt und das Internet zerfaellt in Staub (0.3 %),")
print("   waehrend 10 % ZUFAELLIGE Ausfaelle es kaum jucken (0.84).")
print("   Das ist die direkte Folge der scale-free-Struktur.")

## 6 · Link Prediction: welche Peerings fehlen?

**Aufgabe:** Der gemessene Graph ist unvollständig. Welche Kanten *müsste* es geben? Der
Charme: das braucht **keine Labels** — der Graph labelt sich selbst.

**Vorgehen:** 10 % der Kanten entfernen (= **positive** Testbeispiele), gleich viele
nicht-existente Paare ziehen (= **negative**), und prüfen, ob Heuristiken sie trennen.

> ⚠️ **Falle 1 (Leakage):** Die Heuristiken werden auf dem **Restgraphen** $G_{\text{train}}$
> berechnet — **nachdem** die Testkanten entfernt wurden. Würde man sie auf dem vollen $G$
> rechnen, „sähe" Common Neighbors die gesuchte Kante bereits: die Ergebnisse wären grandios
> und komplett wertlos (Skript 2.5).

In [ ]:
kanten = list(G.edges())
random.Random(SEED).shuffle(kanten)
n_test = int(0.10 * len(kanten))
test_pos = kanten[:n_test]

# WICHTIG: Testkanten entfernen, BEVOR Features berechnet werden
G_train = G.copy()
G_train.remove_edges_from(test_pos)
print(f"G_train: {G_train.number_of_edges():,} Kanten "
      f"({n_test:,} zum Testen entfernt)")

# gleich viele NICHT-existente Paare als negative Beispiele
knoten = list(G.nodes())
test_neg, gesehen = [], set()
while len(test_neg) < n_test:
    u, v = random.Random(SEED + len(test_neg)).sample(knoten, 2)
    if u != v and not G.has_edge(u, v) and (u, v) not in gesehen:
        gesehen.add((u, v)); test_neg.append((u, v))
print(f"Negative Beispiele: {len(test_neg):,}")

### 6.1 · Die Heuristiken

**Deine Aufgabe:** Implementiere die vier klassischen Maße. Mit $N(u)$ = Nachbarn von $u$ im
**Trainingsgraphen**:

| Heuristik | Formel |
|---|---|
| Common Neighbors | $\|N(u)\cap N(v)\|$ |
| Jaccard | $\frac{\|N(u)\cap N(v)\|}{\|N(u)\cup N(v)\|}$ |
| **Adamic-Adar** | $\sum_{w\in N(u)\cap N(v)}\frac{1}{\log d_w}$ |
| Preferential Attachment | $d_u\cdot d_v$ |

*(Achtung bei Adamic-Adar: $\log d_w$ kann 0 sein, wenn $d_w=1$ → Division durch Null abfangen.)*

In [ ]:
def nachbarn(H, u):
    return set(H[u]) if u in H else set()

def common_neighbors(H, u, v):
    # TODO
    raise NotImplementedError

def jaccard(H, u, v):
    # TODO: |Schnitt| / |Vereinigung|; leere Vereinigung -> 0.0
    raise NotImplementedError

def adamic_adar(H, u, v):
    # TODO: Summe ueber gemeinsame Nachbarn w von 1/log(grad(w));
    #       grad(w) <= 1 ueberspringen (log(1)=0 -> Division durch Null!)
    raise NotImplementedError

def preferential_attachment(H, u, v):
    # TODO
    raise NotImplementedError

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

HEURISTIKEN = {
    "Common Neighbors": common_neighbors,
    "Jaccard": jaccard,
    "Adamic-Adar": adamic_adar,
    "Preferential Attachment": preferential_attachment,
}

y = np.r_[np.ones(len(test_pos)), np.zeros(len(test_neg))]
paare = test_pos + test_neg

print(f"{'Heuristik':26s} {'ROC-AUC':>9s} {'PR-AUC':>8s}")
for name, fn in HEURISTIKEN.items():
    s = np.array([fn(G_train, u, v) for u, v in paare])
    print(f"{name:26s} {roc_auc_score(y, s):9.4f} {average_precision_score(y, s):8.4f}")
print("\n(Zufall waere ROC-AUC 0.5 und PR-AUC 0.5 - die Testmenge ist 1:1 balanciert.)")
print("Ueberraschung: Preferential Attachment gewinnt?! Das schauen wir uns genauer an...")

### 6.2 · Falle 3: Der Grad-Confound 🕵️

**Preferential Attachment** — die *dümmste* der vier Heuristiken (sie schaut nur auf die Grade
und ignoriert die Nachbarschaft völlig) — hat gerade **gewonnen**. Das sollte dich misstrauisch
machen.

**Verdacht:** Unsere **negativen** Beispiele sind *uniform zufällig* gezogene Knotenpaare. In
einem scale-free-Graphen ist ein zufälliger Knoten fast sicher ein **Blatt** (Median-Grad 2!) —
ein zufälliges Paar ist also fast immer **Blatt × Blatt**. Echte Kanten hingegen betreffen
überdurchschnittlich oft **Hubs**. PA muss also gar keine „Struktur" erkennen: es unterscheidet
schlicht *Hub-beteiligt* von *Blatt×Blatt*. Prüfen wir das.

In [ ]:
gradprodukt = lambda paare_: np.array([G.degree(u) * G.degree(v) for u, v in paare_])
print("Median des Gradprodukts d_u * d_v:")
print(f"  echte Kanten (positiv) : {np.median(gradprodukt(test_pos)):>8.0f}")
print(f"  Zufallspaare (negativ) : {np.median(gradprodukt(test_neg)):>8.0f}")
print("  -> Faktor >100 Unterschied. PA trennt die beiden Gruppen fast trivial,")
print("     ohne irgendetwas ueber die tatsaechliche Nachbarschaft zu wissen.\n")

# Gegenprobe: GRAD-GEMATCHTE Negativbeispiele.
# Zu jeder echten Kante (u0,v0) ziehen wir ein NICHT-Paar mit moeglichst gleichen Graden.
_rnd = random.Random(SEED)
bins = {}
for k, d in G.degree():
    bins.setdefault(d, []).append(k)
grad_werte = np.array(sorted(bins))

def ziehe_knoten_mit_grad(d):
    g = int(grad_werte[np.argmin(np.abs(grad_werte - d))])
    return _rnd.choice(bins[g])

neg_gematcht = []
for (u0, v0) in test_pos:
    du, dv = G.degree(u0), G.degree(v0)
    for _ in range(40):
        u, v = ziehe_knoten_mit_grad(du), ziehe_knoten_mit_grad(dv)
        if u != v and not G.has_edge(u, v):
            neg_gematcht.append((u, v)); break

print(f"grad-gematchte Negativbeispiele: {len(neg_gematcht):,}")
print(f"Median Gradprodukt jetzt: positiv {np.median(gradprodukt(test_pos)):.0f} | "
      f"negativ {np.median(gradprodukt(neg_gematcht)):.0f}  <- jetzt vergleichbar\n")

y2 = np.r_[np.ones(len(test_pos)), np.zeros(len(neg_gematcht))]
paare2 = test_pos + neg_gematcht
print(f"{'Heuristik':26s} {'uniform':>9s} {'grad-gematcht':>14s}")
for name, fn in HEURISTIKEN.items():
    s1 = np.array([fn(G_train, u, v) for u, v in paare])
    s2 = np.array([fn(G_train, u, v) for u, v in paare2])
    print(f"{name:26s} {roc_auc_score(y, s1):9.4f} {roc_auc_score(y2, s2):14.4f}")

**Das Ergebnis ist eindeutig — und ein Lehrstück:**

| Heuristik | uniform Negative | **grad-gematcht** |
|---|---|---|
| Common Neighbors | 0,719 | 0,559 |
| Jaccard | 0,698 | 0,565 |
| **Adamic-Adar** | 0,724 | **0,566** ← jetzt der Beste |
| **Preferential Attachment** | **0,763** ← „Sieger" | **0,406** ← schlechter als Raten! |

**Preferential Attachment stürzt von 0,763 auf 0,406** — also auf Zufallsniveau und darunter.
Sein gesamter „Vorsprung" war **kein Signal, sondern ein Artefakt der Negativ-Auswahl**. Sobald
positive und negative Paare dieselbe Gradverteilung haben, bleibt von PA **nichts** übrig, und
**Adamic-Adar** ist erwartungsgemäß die beste Heuristik.

Gleichzeitig fallen *alle* Verfahren auf ~0,56: **die Aufgabe ist in Wahrheit viel schwerer**,
als die erste Tabelle suggeriert hat.

> **Die Lektion:** Nicht nur das *Modell* kann täuschen — schon die **Konstruktion der
> Testmenge** entscheidet, was man misst. Uniform gezogene Negativbeispiele sind der Standard in
> unzähligen Link-Prediction-Papers. Sie messen zu einem großen Teil nur, ob ein Verfahren Grade
> lesen kann. Zusammen mit der Basisraten-Falle (unten) heißt das: **Link-Prediction-Zahlen aus
> der Literatur sind mit großer Vorsicht zu genießen.**

## 7 · Fazit

**Was du reproduziert hast — auf echten Daten:**
1. **Scale-free**: Median-Grad 2, aber ein Hub mit **2312** Nachbarn. Kein typischer Grad.
2. **Der Exponent**: MLE ≈ **2,08–2,12** (= Literaturwert ~2,1 ✓), naiver log-log-Fit ≈ **1,1**
   — **Faktor 2 daneben**. Die bequeme Methode ist die falsche.
3. **Robustheit**: 10 % zufälliger Ausfall → 84 % bleiben verbunden. **5 % gezielt → 0,3 %.**
   Dieselbe Struktur, die das Internet gegen Pannen immunisiert, macht es gegen Angreifer
   fragil.
4. **Link Prediction** funktioniert ohne jedes Label — **Adamic-Adar** ist die stärkste
   Heuristik, weil ihre Gewichtung $1/\log d_w$ genau das Richtige tut: ein gemeinsamer Nachbar
   mit Grad 3 ist ein starkes Indiz, ein Tier-1-Hub mit Grad 2000 (mit dem *alle* verbunden
   sind) sagt fast nichts.

> ### ⚠️ Falle 2: die Basisrate — schon wieder
> Unsere Testmenge ist **1:1 balanciert** (gleich viele positive wie negative Paare). Bequem —
> aber die **Realität** ist: 22 002 Kanten unter **57 Mio.** möglichen Paaren, also
> $\pi\approx 3{,}9\cdot10^{-4}$. Nach dem **Base-Rate-Fallacy** (Modul 15!) heißt eine ROC-AUC
> von 0,95 auf balancierten Daten im echten Einsatz — alle Paare durchprobieren — **fast nur
> Fehlalarme**. Balanciertes Sampling ist Standard in der Literatur; man muss nur ehrlich
> dazusagen, was es **nicht** zeigt.

### Mini-Aufgaben
1. Berechne die echte Basisrate $\pi$ und schätze mit der Bayes-Formel aus Modul 15 (P02) den
   **PPV** von Adamic-Adar bei einem realistischen Betriebspunkt. Ernüchternd?
2. Berechne die Heuristiken **auf dem vollen `G`** statt auf `G_train` (also mit Leakage) —
   wie fantastisch werden die Zahlen?
3. Entferne statt 10 % nur 1 % der Kanten. Werden die Heuristiken besser? Warum?
4. Vergleiche die Robustheitskurve mit einem **Zufallsgraphen** gleicher Knoten-/Kantenzahl
   (`nx.gnm_random_graph`). Verschwindet die Asymmetrie? (Das ist der eigentliche Beweis, dass
   sie an der **scale-free-Struktur** liegt.)